# 02 — Reconstruction Benchmark

This notebook trains a CNN/UNet wavefront reconstructor on a small
dataset and benchmarks it against the classical modal (SVD) reconstructor.

1. Load dataset
2. Train CNN for a few epochs (quick demo)
3. Run `benchmark_reconstruction` on the test set
4. Plot RMS WFE comparison (classical vs CNN)
5. Plot noise robustness curves
6. Plot dropout robustness curves
7. Show side-by-side phase maps: ground truth / classical / CNN
8. Compute and display a Strehl ratio table


In [1]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml

from sim.dataset_gen import generate_dataset, load_dataset, SHWFSDataset
from sim.shwfs import SHWFSSensor
from reconstruction.zernike import zernike_basis
from reconstruction.classical import ModalReconstructor
from reconstruction.train import train_model, load_trained_model, predict_batch
from reconstruction.benchmark import (
    benchmark_reconstruction, benchmark_noise_robustness, benchmark_dropout_robustness,
    print_benchmark_summary,
)
from viz.plot_utils import plot_phase_map, plot_noise_robustness, plot_benchmark_table
from sim.phase_screen import get_aperture_mask, apply_aperture_mask, zernike_reconstruct, compute_strehl_ratio

with open("../config.yaml") as f:
    config = yaml.safe_load(f)

# Use a smaller, fast configuration for this demo notebook
config["sim"]["n_frames"] = 300
config["sim"]["grid_size"] = 64
config["reconstruction"]["n_epochs"] = 5
config["reconstruction"]["batch_size"] = 16

sim_cfg = config["sim"]
N = sim_cfg["grid_size"]
n_zernike = sim_cfg["n_zernike"]


## 1. Load (or generate) dataset


In [2]:
os.makedirs("../data", exist_ok=True)
os.makedirs("../models", exist_ok=True)

dataset_path = "../data/dataset_demo.h5"
if not os.path.exists(dataset_path):
    generate_dataset(config, n_frames=sim_cfg["n_frames"], output_path=dataset_path, seed=42)

data = load_dataset(dataset_path)
print("slopes:", data["slopes"].shape)
print("zernike_coeffs:", data["zernike_coeffs"].shape)


Generating dataset:   0%|          | 0/300 [00:00<?, ?it/s]

Generating dataset:   2%|▏         | 5/300 [00:00<00:07, 41.17it/s]

Generating dataset:   3%|▎         | 10/300 [00:00<00:07, 40.24it/s]

Generating dataset:   5%|▌         | 15/300 [00:00<00:07, 40.55it/s]

Generating dataset:   7%|▋         | 20/300 [00:00<00:06, 40.53it/s]

...

Generating dataset:  96%|█████████▌| 288/300 [00:06<00:00, 41.66it/s]

Generating dataset:  98%|█████████▊| 293/300 [00:07<00:00, 40.72it/s]

Generating dataset:  99%|█████████▉| 298/300 [00:07<00:00, 40.99it/s]

Generating dataset: 100%|██████████| 300/300 [00:07<00:00, 41.16it/s]

slopes: (300, 2, 10, 10)
zernike_coeffs: (300, 36)


## 2. Train CNN/UNet reconstructor (quick demo)


In [3]:
cnn_save_path = "../models/cnn_reconstructor_demo.pt"
history = train_model(config, dataset_path, cnn_save_path)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="train loss")
ax.plot(history["val_loss"], label="val loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("CNN training curve")
ax.legend()
plt.show()


Training CNN reconstructor:   0%|          | 0/5 [00:00<?, ?it/s]

Training CNN reconstructor:   0%|          | 0/5 [00:01<?, ?it/s, train_loss=0.69095, val_loss=0.72686, val_rms_nm=74.58]

Training CNN reconstructor:  20%|██        | 1/5 [00:01<00:04,  1.05s/it, train_loss=0.69095, val_loss=0.72686, val_rms_nm=74.58]

Training CNN reconstructor:  20%|██        | 1/5 [00:02<00:04,  1.05s/it, train_loss=0.53988, val_loss=0.53393, val_rms_nm=63.90]

Training CNN reconstructor:  40%|████      | 2/5 [00:02<00:03,  1.03s/it, train_loss=0.53988, val_loss=0.53393, val_rms_nm=63.90]

...

Training CNN reconstructor:  60%|██████    | 3/5 [00:04<00:02,  1.00s/it, train_loss=0.31888, val_loss=0.25711, val_rms_nm=44.34]

Training CNN reconstructor:  80%|████████  | 4/5 [00:04<00:00,  1.00it/s, train_loss=0.31888, val_loss=0.25711, val_rms_nm=44.34]

Training CNN reconstructor:  80%|████████  | 4/5 [00:04<00:00,  1.00it/s, train_loss=0.27552, val_loss=0.24060, val_rms_nm=42.89]

Training CNN reconstructor: 100%|██████████| 5/5 [00:04<00:00,  1.01it/s, train_loss=0.27552, val_loss=0.24060, val_rms_nm=42.89]

Training CNN reconstructor: 100%|██████████| 5/5 [00:05<00:00,  1.00s/it, train_loss=0.27552, val_loss=0.24060, val_rms_nm=42.89]

## 3. Benchmark: classical vs CNN reconstruction


In [4]:
df_recon = benchmark_reconstruction(dataset_path, cnn_save_path, config, n_test_frames=100)
print_benchmark_summary(df_recon)
df_recon.head()


Benchmark summary
          frame  rms_classical   rms_cnn  strehl_classical  strehl_cnn
mean  49.500000       2.123607  0.495532          0.011367    0.782051
std   29.011492       0.064331  0.023632          0.003110    0.018273
min    0.000000       1.991278  0.454384          0.006171    0.751888
max   99.000000       2.255628  0.534011          0.018964    0.813455


,frame,rms_classical,rms_cnn,strehl_classical,strehl_cnn
0,0,2.206271,0.480017,0.007692,0.794203
1,1,2.201404,0.489251,0.007858,0.787126
2,2,2.214690,0.492078,0.007411,0.784946
3,3,2.224145,0.498243,0.007106,0.780168
4,4,2.255628,0.493918,0.006171,0.783523


## 4. Plot RMS WFE comparison


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_recon["frame"], df_recon["rms_classical"], label="classical (modal/SVD)")
ax.plot(df_recon["frame"], df_recon["rms_cnn"], label="CNN/UNet")
ax.set_xlabel("frame")
ax.set_ylabel("RMS WFE (radians)")
ax.set_title("RMS wavefront error: classical vs CNN")
ax.legend()
plt.show()


## 5. Noise robustness curves


In [6]:
df_noise = benchmark_noise_robustness(dataset_path, cnn_save_path, config, noise_levels=[1, 3, 5, 10, 20, 40])

fig, ax = plt.subplots(figsize=(6, 4))
plot_noise_robustness(df_noise["noise_level"].values, df_noise["rms_classical"].values, df_noise["rms_cnn"].values, ax=ax)
plt.show()

df_noise


,noise_level,rms_classical,rms_cnn
0,1,2.487158,0.318454
1,3,2.490390,0.319711
2,5,2.496839,0.322387
3,10,2.526833,0.337528
4,20,2.643481,0.424380
5,40,3.071066,0.826456


## 6. Dropout robustness curves


In [7]:
df_dropout = benchmark_dropout_robustness(dataset_path, cnn_save_path, config)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df_dropout["dropout_frac"], df_dropout["rms_classical"], "o-", label="classical")
ax.plot(df_dropout["dropout_frac"], df_dropout["rms_cnn"], "s-", label="CNN")
ax.set_xlabel("subaperture dropout fraction")
ax.set_ylabel("RMS WFE (radians)")
ax.set_title("Dropout robustness")
ax.legend()
plt.show()

df_dropout


,dropout_frac,rms_classical,rms_cnn
0,0.0,2.399764,0.321030
1,0.1,2.154654,0.309481
2,0.2,1.887877,0.303669
3,0.3,1.607208,0.307935
4,0.5,1.161443,0.344562


## 7. Side-by-side phase maps: ground truth / classical / CNN


In [8]:
sensor = SHWFSSensor(
    n_subapertures=sim_cfg["n_subapertures"],
    pixels_per_subaperture=sim_cfg["detector_pixels_per_subaperture"],
    focal_length=sim_cfg["focal_length_m"],
    pitch=sim_cfg["mla_pitch_m"],
    wavelength=sim_cfg["wavelength_m"],
)
basis = zernike_basis(n_zernike, N)
modal_recon = ModalReconstructor(sensor, basis, n_zernike, config["reconstruction"]["svd_condition_number"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model = load_trained_model(cnn_save_path, config, device)

mask = get_aperture_mask(N, "circular")

frame_idx = 0
sx, sy = data["slopes"][frame_idx, 0], data["slopes"][frame_idx, 1]
true_coeffs = data["zernike_coeffs"][frame_idx]
true_phase = apply_aperture_mask(data["phase_maps"][frame_idx], "circular")

classical_coeffs = modal_recon.reconstruct(sx, sy)
classical_phase = zernike_reconstruct(classical_coeffs, (N, N), pixel_scale=1.0)
classical_phase = apply_aperture_mask(classical_phase, "circular")

cnn_coeffs = predict_batch(cnn_model, data["slopes"][frame_idx:frame_idx+1].astype(np.float32), device)[0]
cnn_phase = zernike_reconstruct(cnn_coeffs, (N, N), pixel_scale=1.0)
cnn_phase = apply_aperture_mask(cnn_phase, "circular")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_phase_map(true_phase, "Ground truth", mask=mask, ax=axes[0])
plot_phase_map(classical_phase, "Classical (modal/SVD)", mask=mask, ax=axes[1])
plot_phase_map(cnn_phase, "CNN/UNet", mask=mask, ax=axes[2])
plt.tight_layout()
plt.show()


## 8. Strehl ratio table


In [9]:
summary_table = df_recon[["frame", "rms_classical", "rms_cnn", "strehl_classical", "strehl_cnn"]].head(10)
styled = plot_benchmark_table(summary_table)
styled


,frame,rms_classical,rms_cnn,strehl_classical,strehl_cnn,improvement
0,0,2.206271,0.480017,0.007692,0.794203,1.726254
1,1,2.201404,0.489251,0.007858,0.787126,1.712153
2,2,2.214690,0.492078,0.007411,0.784946,1.722612
3,3,2.224145,0.498243,0.007106,0.780168,1.725902
4,4,2.255628,0.493918,0.006171,0.783523,1.761710
5,5,2.219678,0.503148,0.007248,0.776346,1.716530
6,6,2.159530,0.515349,0.009433,0.766758,1.644182
7,7,2.178500,0.509789,0.008688,0.771141,1.668711
8,8,2.112470,0.515097,0.011533,0.766957,1.597373
9,9,2.151798,0.514577,0.009752,0.767368,1.637222


In [10]:
print(f"Mean Strehl (classical): {df_recon['strehl_classical'].mean():.4f}")
print(f"Mean Strehl (CNN):       {df_recon['strehl_cnn'].mean():.4f}")
print(f"Mean RMS WFE (classical): {df_recon['rms_classical'].mean():.4f} rad")
print(f"Mean RMS WFE (CNN):       {df_recon['rms_cnn'].mean():.4f} rad")


Mean Strehl (classical): 0.0114
Mean Strehl (CNN):       0.7821
Mean RMS WFE (classical): 2.1236 rad
Mean RMS WFE (CNN):       0.4955 rad
